# Model Finetuning
We will finetune the model in the following steps:
- Take in the finetuned dataset
- Convert to the proper format (tensors)
- Then finetune the model based

Some imports for libaries we will use and helper classes for finetuning the model. Instead of writing our own training loop and evaluation function, we will be using the given functions from pytorch directly. 

In [1]:
import torch
from torchvision.models.detection import fasterrcnn_resnet50_fpn
import json # For our manually labeled finetune train data
from PIL import Image
import torchvision.transforms as T
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torch.utils.data import Dataset
import os
import sys

base = "https://raw.githubusercontent.com/pytorch/vision/main/references/detection"
files = ["engine.py", "utils.py", "coco_utils.py", "coco_eval.py", "transforms.py"]

for f in files:
    os.system(f"curl -o finetune_helpers/{f} {base}/{f}")

sys.path.append("./finetune_helpers")
from finetune_helpers.engine import train_one_epoch
from finetune_helpers import utils

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  4063  100  4063    0     0  18848      0 --:--:-- --:--:-- --:--:-- 18810
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  8388  100  8388    0     0  46155      0 --:--:-- --:--:-- --:--:-- 46342
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  8409  100  8409    0     0  38158      0 --:--:-- --:--:-- --:--:-- 38222
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  6447  100  6447    0     0  30694      0 --:--:-- --:--:-- --:--:-- 30846
  % Total    % Received % Xferd  Average Speed   Tim

# Steps 1 & 2
First we will take in the finetuned dataset (*annotations.json*) and convert it into a list of dictionaries where each element of the dictinoary is either "boxes", i.e. the location of boxes on our classified images, or "labels", i.e. 1 since we are classifying tree (1) and not tree (0). 
We do this in the __init__ function.

Since we will be working with the data when training/finetuning we create a class that inherits from Dataset. 

In [2]:
class TreeDataset(Dataset):
    def __init__(self, annotations_path):
        self.root_path = annotations_path
        with open(self.root_path + "annotations.json") as f:
            all_annotations = json.load(f)

        self.annotations = []
        for a in all_annotations:
            if (all_annotations[a] != []): # drop pictures with no trees
                self.annotations.append((a, all_annotations[a]))
        self.transform = T.ToTensor()

    def __len__(self):
        return len(self.annotations)  # how many images total

    def __getitem__(self, idx):
        # print(self.annotations)
        # print(self.annotations[1])
        image_name, boxes = self.annotations[idx]
        img = Image.open(self.root_path + "raw_images/" + image_name).convert("RGB")
        img_tensor = self.transform(img)
        
        target = {
            "boxes":  torch.tensor(boxes, dtype=torch.float32),
            "labels": torch.ones(len(boxes), dtype=torch.int64),
            "image_id": torch.tensor([idx])
        }
        
        return img_tensor, target

# Sanity Check:
dataset_full = TreeDataset("./Training Classifier/")
print(f"Total images: {len(dataset_full)}")

Total images: 356


# Step 3
Now we are modifying the pre-trained Faster R-CNN model.  

In particular, we are modifying the classification layer of our Faster R-CNN model. Instead of multiple classes like airplanes, persons, chairs, etc., we will now have **only 2 classes**: tree (1) or no tree (0)

This code is heavily inspired by the [TorchVision Object Detection Finetuning Tutorial](https://docs.pytorch.org/tutorials/intermediate/torchvision_tutorial.html#torchvision-object-detection-finetuning-tutorial)

In [3]:
def get_fine_tune_ready_model():
    # load a model pre-trained on COCO
    model = fasterrcnn_resnet50_fpn(weights="DEFAULT")

    num_classes = 2  # 1 class (tree) + background

    # get number of input features for the classifier (from the conv. layers / pooling)
    in_features = model.roi_heads.box_predictor.cls_score.in_features

    # replace the pre-trained head with a new one
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)

    # Predicted trees that overlap more than 30% are the same tree
    model.roi_heads.nms_thresh = 0.3
    
    # Freezing most parameters (everything but the final head/FC layer)
    for param in model.backbone.parameters():
        param.requires_grad = False
    for param in model.rpn.parameters():
        param.requires_grad = False
    
    return model

# Step 4
Now we will finetune the model. This code is almost directly taken from the [TorchVision Object Detection Finetuning Tutorial](https://docs.pytorch.org/tutorials/intermediate/torchvision_tutorial.html#torchvision-object-detection-finetuning-tutorial) 

In [4]:
# train on the accelerator (mac) or on the CPU, if an accelerator is not available
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

# our dataset has two classes only - background and tree
num_classes = 2
dataset = TreeDataset('./Training Classifier/')

# define training and validation data loaders
data_loader = torch.utils.data.DataLoader(
    dataset,
    batch_size=4,
    shuffle=True,
    collate_fn=utils.collate_fn
)

# get the model using our helper function
model = get_fine_tune_ready_model()

# move model to the right device
model.to(device)

# construct an optimizer
params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.SGD(
    params,
    lr=0.005,
    momentum=0.9,
    weight_decay=0.0005
)

# and a learning rate scheduler
lr_scheduler = torch.optim.lr_scheduler.StepLR(
    optimizer,
    step_size=8,
    gamma=0.3
)

num_epochs = 24

for epoch in range(num_epochs):
    # train for one epoch, printing every 10 iterations
    train_one_epoch(model, optimizer, data_loader, device, epoch, print_freq=10)
    # update the learning rate
    lr_scheduler.step()

print("That's it!")

/Users/paulrabel/tree_detector/model/finetune_helpers/engine.py:30: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=scaler is not None):


Epoch: [0]  [ 0/89]  eta: 0:03:06  lr: 0.000062  loss: 5.3255 (5.3255)  loss_classifier: 0.7613 (0.7613)  loss_box_reg: 0.2620 (0.2620)  loss_objectness: 4.0905 (4.0905)  loss_rpn_box_reg: 0.2117 (0.2117)  time: 2.0942  data: 0.0866
Epoch: [0]  [10/89]  eta: 0:01:57  lr: 0.000629  loss: 5.3255 (5.2389)  loss_classifier: 0.5909 (0.6019)  loss_box_reg: 0.2024 (0.2140)  loss_objectness: 4.1458 (4.1952)  loss_rpn_box_reg: 0.2289 (0.2278)  time: 1.4922  data: 0.0738
Epoch: [0]  [20/89]  eta: 0:01:37  lr: 0.001197  loss: 5.1450 (5.1219)  loss_classifier: 0.5219 (0.5510)  loss_box_reg: 0.1908 (0.2075)  loss_objectness: 4.1931 (4.1354)  loss_rpn_box_reg: 0.2327 (0.2280)  time: 1.3735  data: 0.0657
Epoch: [0]  [30/89]  eta: 0:01:22  lr: 0.001765  loss: 4.9822 (5.0789)  loss_classifier: 0.3331 (0.4598)  loss_box_reg: 0.1696 (0.1910)  loss_objectness: 4.1931 (4.1980)  loss_rpn_box_reg: 0.2196 (0.2301)  time: 1.3417  data: 0.0616
Epoch: [0]  [40/89]  eta: 0:01:08  lr: 0.002332  loss: 4.7218 (4.972

# Save weights:
Save the learned weights

In [5]:
torch.save(model.state_dict(), "weights.pt")

# More Adjustments

Currently, we only finetuned the classification layer of our model (the very last layer). Using the following parameters:
- `batch size = 4`
- `learning rate = 0.005`
- `momentum = 0.9`
- `weight decay = 0.0005`
- `step size = 8`
- `gamma = 0.3`
- `epochs = 24`

the classification loss was reduced from $~0.87$ to $~0.08$. 

However, the total loss stayed relatively constant, between $4$ and $5$, mainly due to the **loss_objectness** which consistenly is around $4$. **loss_objectness** is measuring the performance of our model determining whether any object is at a location. Since the model was trained on `COCO`, and satellite-view trees are not part of that training data, we must also update weights in the RPN layer of the model. Thus, we proceed with *progressive unfreezing*.

In [6]:
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

num_classes = 2
dataset = TreeDataset('./Training Classifier/')

# define training and validation data loaders
data_loader = torch.utils.data.DataLoader(
    dataset,
    batch_size=4,
    shuffle=True,
    collate_fn=utils.collate_fn
)

# get the model
model = get_fine_tune_ready_model()
model.load_state_dict(torch.load("weights.pt", weights_only=True))
for param in model.rpn.parameters(): # go deeper than before
    param.requires_grad = True
model.to(device)

# construct an optimizer
params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.SGD(
    params,
    lr=0.0005,
    momentum=0.9,
    weight_decay=0.0005
)

# and a learning rate scheduler
lr_scheduler = torch.optim.lr_scheduler.StepLR(
    optimizer,
    step_size=8,
    gamma=0.5
)

num_epochs = 12

for epoch in range(num_epochs):
    # train for one epoch, printing every 10 iterations
    train_one_epoch(model, optimizer, data_loader, device, epoch, print_freq=10)
    # update the learning rate
    lr_scheduler.step()

print("That's it!")

Epoch: [0]  [ 0/89]  eta: 0:03:39  lr: 0.000006  loss: 4.9070 (4.9070)  loss_classifier: 0.0818 (0.0818)  loss_box_reg: 0.1457 (0.1457)  loss_objectness: 4.4023 (4.4023)  loss_rpn_box_reg: 0.2772 (0.2772)  time: 2.4693  data: 0.0785
Epoch: [0]  [10/89]  eta: 0:02:10  lr: 0.000063  loss: 4.2680 (4.3630)  loss_classifier: 0.0754 (0.0701)  loss_box_reg: 0.1263 (0.1186)  loss_objectness: 3.8541 (3.9687)  loss_rpn_box_reg: 0.1969 (0.2056)  time: 1.6489  data: 0.0732
Epoch: [0]  [20/89]  eta: 0:02:20  lr: 0.000120  loss: 3.8565 (3.9286)  loss_classifier: 0.0810 (0.0938)  loss_box_reg: 0.1370 (0.1520)  loss_objectness: 3.5146 (3.4861)  loss_rpn_box_reg: 0.1826 (0.1967)  time: 2.0094  data: 0.0696
Epoch: [0]  [30/89]  eta: 0:02:11  lr: 0.000176  loss: 3.1393 (3.5667)  loss_classifier: 0.1588 (0.1727)  loss_box_reg: 0.2379 (0.1994)  loss_objectness: 2.4247 (2.9924)  loss_rpn_box_reg: 0.1617 (0.2021)  time: 2.5416  data: 0.0932
Epoch: [0]  [40/89]  eta: 0:01:42  lr: 0.000233  loss: 2.3973 (3.247

# Final Step

Now that the **loss_objectness** has decreased (from ~$4$ to now being ~$0.25$) and our total **loss** has also significantly decreased, we can save the final weights.

In [7]:
torch.save(model.state_dict(), "final_weights.pt")